In [1]:
import pandas as pd
import numpy as np
import keras
import tensorflow as tf

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
# get demand data from NTUST (sum from multiple building in NTUST)
power_df = pd.read_csv("../dataset/NTUST.csv", index_col=0, parse_dates=True)
power_df = power_df.asfreq('h') 
power_df = pd.DataFrame(power_df.loc[power_df.index.year!=2020, "KW"].round(2).copy())

In [3]:
power_df.head()

,KW
timestamp,
2019-01-23 00:00:00,359.32
2019-01-23 01:00:00,308.63
2019-01-23 02:00:00,284.17
2019-01-23 03:00:00,252.00
2019-01-23 04:00:00,263.33


In [4]:
weather_df = pd.read_csv("../dataset/NSRDB.csv")
weather_df.index = power_df.index 

In [ ]:
weather_df.head()

,YEAR,MONTH,DAY,HOUR,TEMPERATURE,DEW_POINT,GHI,PRESSURE,WIND_SPEED,RELATIVE HUMIDITY,PRECIPITABLE WATER,CLOUD_TYPE
timestamp,,,,,,,,,,,,
2019-01-23 00:00:00,2019,1,23,0,14.3,8.8,0,1032,4.1,69.69,1.4,0
2019-01-23 01:00:00,2019,1,23,1,14.2,8.8,0,1031,4.1,70.21,1.4,3
2019-01-23 02:00:00,2019,1,23,2,14.1,8.9,0,1031,4.1,71.05,1.4,3
2019-01-23 03:00:00,2019,1,23,3,14.1,9.0,0,1031,4.1,71.62,1.4,0
2019-01-23 04:00:00,2019,1,23,4,14.1,9.2,0,1031,4.0,72.27,1.4,3


In [7]:
weather_df = weather_df[['GHI', 'TEMPERATURE']]

In [8]:
weather_df.head()

,GHI,TEMPERATURE
timestamp,,
2019-01-23 00:00:00,0,14.3
2019-01-23 01:00:00,0,14.2
2019-01-23 02:00:00,0,14.1
2019-01-23 03:00:00,0,14.1
2019-01-23 04:00:00,0,14.1


In [9]:
combined_df = pd.concat([power_df, weather_df], axis=1)

In [10]:
combined_df.head()

,KW,GHI,TEMPERATURE
timestamp,,,
2019-01-23 00:00:00,359.32,0,14.3
2019-01-23 01:00:00,308.63,0,14.2
2019-01-23 02:00:00,284.17,0,14.1
2019-01-23 03:00:00,252.00,0,14.1
2019-01-23 04:00:00,263.33,0,14.1


In [11]:
combined_df['date'] = combined_df.index.strftime('%Y-%m-%d')
combined_df['time'] = combined_df.index.strftime('%H:%M:%S')

In [12]:
combined_df.head()

,KW,GHI,TEMPERATURE,date,time
timestamp,,,,,
2019-01-23 00:00:00,359.32,0,14.3,2019-01-23,00:00:00
2019-01-23 01:00:00,308.63,0,14.2,2019-01-23,01:00:00
2019-01-23 02:00:00,284.17,0,14.1,2019-01-23,02:00:00
2019-01-23 03:00:00,252.00,0,14.1,2019-01-23,03:00:00
2019-01-23 04:00:00,263.33,0,14.1,2019-01-23,04:00:00


In [15]:
new_df = combined_df[['date', 'time', 'KW', 'GHI', 'TEMPERATURE']].copy()

In [16]:
new_df.head()

,date,time,KW,GHI,TEMPERATURE
timestamp,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,359.32,0,14.3
2019-01-23 01:00:00,2019-01-23,01:00:00,308.63,0,14.2
2019-01-23 02:00:00,2019-01-23,02:00:00,284.17,0,14.1
2019-01-23 03:00:00,2019-01-23,03:00:00,252.00,0,14.1
2019-01-23 04:00:00,2019-01-23,04:00:00,263.33,0,14.1


In [17]:
new_df.to_csv('./new_dataset.csv', index=False)

In [18]:
temperature_df = new_df.copy()
temperature_df['OT'] = temperature_df['TEMPERATURE'].copy()

In [19]:
temperature_df.head()

,date,time,KW,GHI,TEMPERATURE,OT
timestamp,,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,359.32,0,14.3,14.3
2019-01-23 01:00:00,2019-01-23,01:00:00,308.63,0,14.2,14.2
2019-01-23 02:00:00,2019-01-23,02:00:00,284.17,0,14.1,14.1
2019-01-23 03:00:00,2019-01-23,03:00:00,252.00,0,14.1,14.1
2019-01-23 04:00:00,2019-01-23,04:00:00,263.33,0,14.1,14.1


In [20]:
temperature_df = temperature_df.drop(columns=['TEMPERATURE'])

In [21]:
temperature_df.head()

,date,time,KW,GHI,OT
timestamp,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,359.32,0,14.3
2019-01-23 01:00:00,2019-01-23,01:00:00,308.63,0,14.2
2019-01-23 02:00:00,2019-01-23,02:00:00,284.17,0,14.1
2019-01-23 03:00:00,2019-01-23,03:00:00,252.00,0,14.1
2019-01-23 04:00:00,2019-01-23,04:00:00,263.33,0,14.1


In [22]:
temperature_df.to_csv('./temperature_dataset.csv', index=False)

In [23]:
ghi_df = new_df.copy()
ghi_df['OT'] = ghi_df['GHI'].copy()

In [24]:
ghi_df.head()

,date,time,KW,GHI,TEMPERATURE,OT
timestamp,,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,359.32,0,14.3,0
2019-01-23 01:00:00,2019-01-23,01:00:00,308.63,0,14.2,0
2019-01-23 02:00:00,2019-01-23,02:00:00,284.17,0,14.1,0
2019-01-23 03:00:00,2019-01-23,03:00:00,252.00,0,14.1,0
2019-01-23 04:00:00,2019-01-23,04:00:00,263.33,0,14.1,0


In [25]:
ghi_df = ghi_df.drop(columns=['GHI'])

In [26]:
ghi_df.head()

,date,time,KW,TEMPERATURE,OT
timestamp,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,359.32,14.3,0
2019-01-23 01:00:00,2019-01-23,01:00:00,308.63,14.2,0
2019-01-23 02:00:00,2019-01-23,02:00:00,284.17,14.1,0
2019-01-23 03:00:00,2019-01-23,03:00:00,252.00,14.1,0
2019-01-23 04:00:00,2019-01-23,04:00:00,263.33,14.1,0


In [27]:
ghi_df.to_csv('./ghi_dataset.csv', index=False)

In [28]:
kw_df = new_df.copy()
kw_df['OT'] = kw_df['KW'].copy()

In [29]:
kw_df.head()

,date,time,KW,GHI,TEMPERATURE,OT
timestamp,,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,359.32,0,14.3,359.32
2019-01-23 01:00:00,2019-01-23,01:00:00,308.63,0,14.2,308.63
2019-01-23 02:00:00,2019-01-23,02:00:00,284.17,0,14.1,284.17
2019-01-23 03:00:00,2019-01-23,03:00:00,252.00,0,14.1,252.00
2019-01-23 04:00:00,2019-01-23,04:00:00,263.33,0,14.1,263.33


In [30]:
kw_df = kw_df.drop(columns=['KW'])

In [31]:
kw_df.head()

,date,time,GHI,TEMPERATURE,OT
timestamp,,,,,
2019-01-23 00:00:00,2019-01-23,00:00:00,0,14.3,359.32
2019-01-23 01:00:00,2019-01-23,01:00:00,0,14.2,308.63
2019-01-23 02:00:00,2019-01-23,02:00:00,0,14.1,284.17
2019-01-23 03:00:00,2019-01-23,03:00:00,0,14.1,252.00
2019-01-23 04:00:00,2019-01-23,04:00:00,0,14.1,263.33


In [32]:
kw_df.to_csv('./kw_dataset.csv', index=False)